<a href="https://colab.research.google.com/github/pujisubarkah/replikasi_DAiSEE-Towards-User-Engagement-Recognition-in-the-Wild/blob/main/Notebook_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LRCN (Long-term Recurrent Convolutional Network) di paper ini adalah :

CNN digunakan untuk memahami isi setiap frame video, sedangkan LSTM digunakan untuk memahami perubahan antar frame sehingga dapat mengenali engagement sebagai sebuah urutan waktu (temporal sequence).

menggunakan Long-Term Recurrent Convolutional Network (LRCN),
tidak menggunakan optical flow,
hanya menggunakan informasi RGB,
video dibangun kembali dengan mengambil setiap frame selang satu frame (every alternate frame) karena affective state tidak berubah dalam waktu kurang dari 30 ms,
output layer diubah menjadi 4 kelas,
model dilatih penuh pada dataset DAiSEE.
Paper tidak menuliskan detail implementasi LRCN, misalnya:

hidden size LSTM
jumlah layer LSTM
dropout
optimizer
learning rate

Karena paper hanya menyebut menggunakan implementasi vanilla dengan sedikit hyperparameter tuning.

Jadi pada bagian ini kita harus memilih nilai yang wajar dan menjelaskannya pada metodologi sebagai implementation details, bukan mengklaim bahwa nilai tersebut berasal dari paper.

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import pandas as pd

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import kagglehub

In [ ]:
dataset_path = kagglehub.dataset_download(
    "olgaparfenova/daisee"
)

print(dataset_path)

Using Colab cache for faster access to the 'daisee' dataset.
/kaggle/input/daisee


In [ ]:
DAISEE_PATH = os.path.join(dataset_path, "DAiSEE")

DATA_PATH = os.path.join(DAISEE_PATH, "DataSet")
LABEL_PATH = os.path.join(DAISEE_PATH, "Labels")

TRAIN_PATH = os.path.join(DATA_PATH, "Train")
VAL_PATH = os.path.join(DATA_PATH, "Validation")
TEST_PATH = os.path.join(DATA_PATH, "Test")

In [ ]:
# Load DataFrames
train_df = pd.read_csv(
    os.path.join(LABEL_PATH, "TrainLabels.csv")
)

val_df = pd.read_csv(
    os.path.join(LABEL_PATH, "ValidationLabels.csv")
)

test_df = pd.read_csv(
    os.path.join(LABEL_PATH, "TestLabels.csv")
)

print(f"Train DataFrame shape: {train_df.shape}")
print(f"Validation DataFrame shape: {val_df.shape}")
print(f"Test DataFrame shape: {test_df.shape}")

Train DataFrame shape: (5358, 5)
Validation DataFrame shape: (1429, 5)
Test DataFrame shape: (1784, 5)


In [ ]:
#load video
def load_video(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        frames.append(frame)

    cap.release()

    return frames

In [ ]:
# Set device for PyTorch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

Using device: cuda


In [ ]:
def uniform_sampling(frames, num_frames=30):

    indices = np.linspace(
        0,
        len(frames)-1,
        num_frames,
        dtype=int
    )

    sampled = [frames[i] for i in indices]

    return sampled

In [ ]:
# Define 'frames' by loading a sample video
sample_clip_id = train_df.iloc[0]['ClipID']
subject = sample_clip_id[:6]
video_path = os.path.join(TRAIN_PATH, subject, sample_clip_id.replace('.avi', ''), sample_clip_id)
frames = load_video(video_path)

sampled_frames = uniform_sampling(frames)

print(f"Original number of frames: {len(frames)}")
print(f"Sampled number of frames: {len(sampled_frames)}")

Original number of frames: 300
Sampled number of frames: 30


In [ ]:
def resize_frames(frames, size=(299, 299)):

    return [
        cv2.resize(frame, size)
        for frame in frames
    ]

In [ ]:
def preprocess_video(video_path):

    frames = load_video(video_path)

    frames = uniform_sampling(frames)

    frames = resize_frames(frames)

    frames = normalize_frames(frames)

    return frames

In [ ]:
# Helper function to load video frames
def load_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    return frames

In [ ]:
def normalize_frames(frames):

    frames = np.array(frames, dtype=np.float32)

    frames /= 255.0

    return frames

In [ ]:
def get_video_path(split_path, clip_name):

    clip_id = os.path.splitext(clip_name)[0]

    subject = clip_id[:6]

    video_path = os.path.join(
        split_path,
        subject,
        clip_id,
        clip_name
    )

    return video_path

In [ ]:
cnn = models.inception_v3(
    weights=models.Inception_V3_Weights.DEFAULT
)

cnn.aux_logits = False
cnn.AuxLogits = None

cnn.fc = torch.nn.Identity()

cnn.eval()

cnn.to(device)

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [ ]:
dummy = torch.randn(
    1,
    3,
    299,
    299
).to(device)

with torch.no_grad():

    feature = cnn(dummy)

print(feature.shape)

torch.Size([1, 2048])


In [ ]:
def extract_features(video_path, cnn, device):

    # Preprocessing video
    frames = preprocess_video(video_path)

    # (30,299,299,3) -> (30,3,299,299)
    frames = np.transpose(frames, (0,3,1,2))

    # NumPy -> Torch Tensor
    frames = torch.tensor(
        frames,
        dtype=torch.float32
    ).to(device)

    cnn.eval()

    with torch.no_grad():

        features = cnn(frames)

    return features.cpu().numpy()

In [ ]:
clip_name = train_df.iloc[0]["ClipID"]

video_path = get_video_path(
    TRAIN_PATH,
    clip_name
)

features = extract_features(
    video_path,
    cnn,
    device
)

print(features.shape)

(30, 2048)


In [ ]:
print(features[0][:10])

[0.48239622 0.17180486 0.3246023  0.40304083 0.06446415 0.45638186
 0.7313727  0.5199145  0.6487782  0.37999547]


In [ ]:
FEATURE_PATH = "/content/features"

os.makedirs(FEATURE_PATH, exist_ok=True)

In [ ]:
clip_id = os.path.splitext(clip_name)[0]

save_path = os.path.join(
    FEATURE_PATH,
    clip_id + ".npy"
)

np.save(save_path, features)

print(save_path)

/content/features/1100011002.npy


In [ ]:
loaded = np.load(save_path)

print(loaded.shape)

(30, 2048)


In [ ]:
FEATURE_ROOT = "/content/features"

TRAIN_FEATURE_PATH = os.path.join(FEATURE_ROOT, "train")
VAL_FEATURE_PATH = os.path.join(FEATURE_ROOT, "validation")
TEST_FEATURE_PATH = os.path.join(FEATURE_ROOT, "test")

os.makedirs(TRAIN_FEATURE_PATH, exist_ok=True)
os.makedirs(VAL_FEATURE_PATH, exist_ok=True)
os.makedirs(TEST_FEATURE_PATH, exist_ok=True)

print("Folder feature berhasil dibuat.")

Folder feature berhasil dibuat.


In [ ]:
from tqdm import tqdm

def extract_split_features(dataframe,
                           split_path,
                           save_path,
                           cnn,
                           device):

    success = 0
    failed = []

    for clip_name in tqdm(dataframe["ClipID"]):

        try:

            video_path = get_video_path(
                split_path,
                clip_name
            )

            features = extract_features(
                video_path,
                cnn,
                device
            )

            clip_id = os.path.splitext(clip_name)[0]

            np.save(
                os.path.join(save_path, clip_id + ".npy"),
                features
            )

            success += 1

        except Exception as e:

            failed.append((clip_name, str(e)))

    print(f"\nSelesai.")
    print(f"Berhasil : {success}")
    print(f"Gagal    : {len(failed)}")

    return failed

In [ ]:
failed_train = extract_split_features(
    dataframe=train_df,
    split_path=TRAIN_PATH,
    save_path=TRAIN_FEATURE_PATH,
    cnn=cnn,
    device=device
)

 18%|█▊        | 946/5358 [08:39<45:10,  1.63it/s]

In [ ]:
failed_test = extract_split_features(
    dataframe=test_df,
    split_path=TEST_PATH,
    save_path=TEST_FEATURE_PATH,
    cnn=cnn,
    device=device
)

In [ ]:
failed_test = extract_split_features(
    dataframe=test_df,
    split_path=TEST_PATH,
    save_path=TEST_FEATURE_PATH,
    cnn=cnn,
    device=device
)

In [ ]:
print("Train :", len(os.listdir(TRAIN_FEATURE_PATH)))
print("Validation :", len(os.listdir(VAL_FEATURE_PATH)))
print("Test :", len(os.listdir(TEST_FEATURE_PATH)))

In [ ]:
import shutil

shutil.make_archive(
    "/content/features",
    "zip",
    FEATURE_ROOT
)

print("features.zip berhasil dibuat.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
FEATURE_ROOT = "/content/drive/MyDrive/DAiSEE_Features"